# ML Model Training Evaluation on Processed UFC Master Data
Will now be performing a train-test split evaluation on multiple machine learning models on the processed data created from the previous notebook. The following ML models be evaluated based on their performance, from which the model with the highest accuracy will be selected:
- Logistic Regression
- Random Forest
- XGBoost
- Support Vector Machine

**Now loading our processed data:**

In [1]:
from pathlib import Path
import pandas as pd

# Always calculate relative to notebook location
NOTEBOOK_DIR = Path.cwd()  # This is /.../ml_core/notebooks/
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent  # Go up 2 levels: /.../ufc-fight-predictor/

# Build the data path to where processed training data source is located (ufc-master-processed.csv)
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "ufc-master-processed.csv"
print(f"📁 Project root: {PROJECT_ROOT}")
print(f"📄 Data path: {DATA_PATH}")

# Load data
df = pd.read_csv(DATA_PATH).drop(columns=["RedFighter", "BlueFighter", "Date"])


# Separating our X and y features
X = df.drop(columns=['Winner_Encoded']) # All our X features
y = df['Winner_Encoded'] # Our target


# Verifying size of original data, training set and test set
print("Total number of rows:")
print(f"Original data: {len(df)}")


📁 Project root: c:\Users\subsi\OneDrive\Desktop\ufc-fight-predictor
📄 Data path: c:\Users\subsi\OneDrive\Desktop\ufc-fight-predictor\data\processed\ufc-master-processed.csv
Total number of rows:
Original data: 6528


**Now evaluating the performance and accuracy of our first model, Logistic Regression**

In [2]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

# First, scale the dataset to be used for the training models that need it (Logistic Regression and Support Vector Machine)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Initializing and training the model
model = LogisticRegression(random_state=42, max_iter=5000)

# Cross-validation for accuracy comparison
score = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')
print(f"Cross-val accuracy scores: {score}")
print(f"Mean accuracy: {score.mean():.2%}")
print(f"Std dev: {score.std():.2%}")

# Fitting separately on full scaled data to extract feature importance
model.fit(X_scaled, y)

# Checking for which features played the biggest role
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', ascending=False)

display(feature_importance)

Cross-val accuracy scores: [0.66003063 0.63782542 0.66232772 0.62988506 0.65900383]
Mean accuracy: 64.98%
Std dev: 1.33%


,feature,coefficient
2,RedExpectedValue,0.389021
26,BlueWeightLbs,0.210795
40,RedWinsByDecisionSplit,0.155113
61,ReachDif,0.148417
14,BlueLosses,0.133347
...,...,...
25,BlueReachCms,-0.147980
37,RedTotalRoundsFought,-0.193716
15,BlueTotalRoundsFought,-0.204379
48,RedWeightLbs,-0.269805


**Now evaluating the accuracy and performance of our second model, Random Forest**

In [3]:
from sklearn.ensemble import RandomForestClassifier

# Will not be using the scaled version of testing set as it's unnecessary
model = RandomForestClassifier(random_state=42, n_estimators=100)

score = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f"Cross-val accuracy scores: {score}")
print(f"Mean accuracy: {score.mean():.2%}")
print(f"Std dev: {score.std():.2%}")

model.fit(X, y)

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

display(feature_importance)

Cross-val accuracy scores: [0.66921899 0.6431853  0.65620214 0.63524904 0.6651341 ]
Mean accuracy: 65.38%
Std dev: 1.29%


,feature,importance
1,BlueOdds,0.049972
3,BlueExpectedValue,0.047910
0,RedOdds,0.046669
2,RedExpectedValue,0.046566
63,SigStrDif,0.027853
...,...,...
103,RedStance_Open Stance,0.000098
69,RWFeatherweightRank,0.000060
102,BlueStance_nan,0.000014
82,BWFeatherweightRank,0.000000


**Now checking for our third model, XGBoost**

In [4]:
from xgboost import XGBClassifier


model = XGBClassifier(random_state=42, eval_metric="logloss")

score = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f"Cross-val accuracy scores: {score}")
print(f"Mean accuracy: {score.mean():.2%}")
print(f"Std dev: {score.std():.2%}")

model.fit(X, y)

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

display(feature_importance)

Cross-val accuracy scores: [0.64624809 0.62404288 0.61255743 0.61302682 0.65823755]
Mean accuracy: 63.08%
Std dev: 1.84%


,feature,importance
2,RedExpectedValue,0.193522
1,BlueOdds,0.040955
87,BMiddleweightRank,0.014285
4,NumberOfRounds,0.013787
68,RWFlyweightRank,0.012301
...,...,...
82,BWFeatherweightRank,0.000000
98,BlueStance_Open Stance,0.000000
96,Gender_Encoded,0.000000
102,BlueStance_nan,0.000000


**Now testing our last model, Support Vector Machine**

In [5]:
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

# Will make use of scaled features, as it's necessary here since its distance-based
model = SVC(random_state=42, probability=False, kernel='linear')

score = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')
print(f"Cross-val accuracy scores: {score}")
print(f"Mean accuracy: {score.mean():.2%}")
print(f"Std dev: {score.std():.2%}")

model.fit(X_scaled, y)

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', ascending=False)

display(feature_importance)

Cross-val accuracy scores: [0.66539051 0.65084227 0.64854518 0.62988506 0.66666667]
Mean accuracy: 65.23%
Std dev: 1.34%


,feature,coefficient
3,BlueExpectedValue,1.131691
0,RedOdds,0.862763
11,BlueAvgTDLanded,0.140787
41,RedWinsByDecisionUnanimous,0.100272
42,RedWinsByKO,0.079069
...,...,...
33,RedAvgTDLanded,-0.119514
45,RedWins,-0.147421
65,AvgTDDif,-0.149281
2,RedExpectedValue,-0.615873


Based on the observations, the model we will move ahead with is the Random Forest Classifier.

| Model | Mean Accuracy | Std Dev |
|---|---|---|
| Logistic Regression | 64.98% | 1.33% |
| Random Forest | 65.38% | 1.29% |
| XGBoost | 63.08% | 1.84% |
| SVM (linear) | 65.23% | 1.34% |

Random Forest Classifier had the highest average accuracy across the 5 folds while also having the lowest standard deviation, indicating both strong and consistent performance. This model will be retrained on the full dataset and exported for use in the backend API.